In [9]:
import numpy as np
import mmap
import os
import struct
import time
import random
import math
import json
from functools import lru_cache

# Limpeza dinâmica de arquivos dat e json antigos
for f in os.listdir('.'):
    if f.startswith("hive_") and (f.endswith(".dat") or f.endswith(".json")):
        try:
            os.remove(f)
        except:
            pass

class GrowableBuffer:
    def __init__(self, path, stride):
        self.path = path
        self.stride = stride
        initial_capacity = 1024 * 1024 * 20 # 20MB
        if not os.path.exists(path):
            with open(path, "wb") as f: f.write(b'\x00' * initial_capacity)
        self.f = open(self.path, "r+b")
        self.mm = mmap.mmap(self.f.fileno(), 0)
        self.capacity = self.mm.size()
        self._tail = 0

    def _ensure_capacity(self, needed):
        if needed <= self.capacity: return
        # CORREÇÃO: max garante que cresce pelo menos o necessário para o novo dado
        new_cap = max(int(self.capacity * 1.5), needed)
        self.mm.close()
        self.f.truncate(new_cap)
        self.mm = mmap.mmap(self.f.fileno(), new_cap)
        self.capacity = new_cap

    def append(self, data_bytes):
        off = self._tail
        self._ensure_capacity(off + len(data_bytes))
        self.mm[off:off+len(data_bytes)] = data_bytes
        self._tail += len(data_bytes)
        return off

    def close(self):
        # CORREÇÃO: Prevenir erro de fechamento duplo e nulos
        if hasattr(self, 'mm') and self.mm is not None:
            try:
                self.mm.close()
            except:
                pass
            self.mm = None
        if hasattr(self, 'f') and self.f is not None:
            try:
                self.f.close()
            except:
                pass
            self.f = None

    def __del__(self):
        try:
            self.close()
        except:
            pass

class DiskHiveStore:
    def __init__(self, base_path, dimension):
        self.base_path = base_path
        self.D = dimension
        self.v_stride = dimension * 4
        self.c_stride = 32
        self.v_buf = GrowableBuffer(f"{base_path}_v.dat", self.v_stride)
        self.c_buf = GrowableBuffer(f"{base_path}_c.dat", self.c_stride)
        self.g_buf = GrowableBuffer(f"{base_path}_g.dat", 4)

        # CORREÇÃO: Persistência do ponteiro de cauda (_tail)
        self.meta_path = f"{base_path}_meta.json"
        if os.path.exists(self.meta_path):
            with open(self.meta_path, "r") as f:
                meta = json.load(f)
                self.v_buf._tail = meta.get("v_tail", 0)
                self.c_buf._tail = meta.get("c_tail", 0)
                self.g_buf._tail = meta.get("g_tail", 0)
        else:
            self.save_meta()

    def save_meta(self):
        # CORREÇÃO: Tratamento para evitar NameError no encerramento do interpretador Python
        try:
            if 'open' in globals() or open is not None:
                with open(self.meta_path, "w") as f:
                    json.dump({
                        "v_tail": self.v_buf._tail,
                        "c_tail": self.c_buf._tail,
                        "g_tail": self.g_buf._tail
                    }, f)
        except:
            pass

    def append_vector(self, vec):
        off = self.v_buf.append(vec.astype(np.float32).tobytes())
        self.save_meta()
        return off // self.v_stride

    def append_graph_edges(self, neighbors):
        off = self.g_buf.append(np.array(neighbors, dtype=np.int32).tobytes())
        self.save_meta()
        return off

    def write_cell_meta(self, cell_id, v_off, n_off, n_count):
        data = struct.pack("<qiiiqi", v_off, 1, n_off, n_count, 0, 1)
        off = cell_id * self.c_stride
        self.c_buf._ensure_capacity(off + self.c_stride)
        self.c_buf.mm[off:off+32] = data
        if off + 32 > self.c_buf._tail: self.c_buf._tail = off + 32
        self.save_meta()

    @lru_cache(maxsize=20000)
    def read_cell_meta(self, cell_id):
        off = cell_id * self.c_stride
        unpacked = struct.unpack("<qiiiqi", self.c_buf.mm[off:off+32])
        return {"v_off": unpacked[0], "n_off": unpacked[2], "n_count": unpacked[3]}

    def read_vector(self, idx):
        return np.frombuffer(self.v_buf.mm, dtype=np.float32, count=self.D, offset=idx * self.v_stride).copy()

    def read_neighbors(self, meta):
        if meta["n_count"] <= 0: return np.array([], dtype=np.int32)
        return np.frombuffer(self.g_buf.mm, dtype=np.int32, count=meta["n_count"], offset=meta["n_off"])

    def close(self):
        try:
            self.save_meta()
        except:
            pass
        self.v_buf.close()
        self.c_buf.close()
        self.g_buf.close()

    def __del__(self):
        try:
            self.close()
        except:
            pass

class HiveBrain:
    def __init__(self, store):
        self.store = store
        self.sentinels_ids = []
        self.sentinels_matrix = None

    def update_sentinels(self, k_sentinels=300):
        total = self.store.c_buf._tail // self.store.c_stride
        if total == 0: return
        
        # 1. Coletar amostra de candidatos para as sentinelas
        n_candidates = min(total, k_sentinels * 5)
        candidates = np.random.choice(total, n_candidates, replace=False)
        
        # 2. Algoritmo Farthest Point Sampling (FPS) para cobertura espacial ideal em RAM
        selected_ids = [candidates[0]]
        selected_vecs = [self.store.read_vector(candidates[0])]
        
        for _ in range(min(k_sentinels - 1, total - 1)):
            cand_vectors = np.array([self.store.read_vector(c) for c in candidates])
            sims = np.dot(cand_vectors, np.array(selected_vecs).T)
            min_sims = np.max(sims, axis=1)
            
            farthest_idx = candidates[np.argmin(min_sims)]
            
            if farthest_idx in selected_ids:
                remaining = list(set(candidates) - set(selected_ids))
                if not remaining: break
                farthest_idx = np.random.choice(remaining)
                
            selected_ids.append(farthest_idx)
            selected_vecs.append(self.store.read_vector(farthest_idx))
            
        self.sentinels_ids = selected_ids
        self.sentinels_matrix = np.array(selected_vecs, dtype=np.float32)

    def search(self, query, beam_width=5, n_entry_points=3):
        # Se as sentinelas não estiverem inicializadas, inicializa
        if self.sentinels_matrix is None or len(self.sentinels_ids) == 0:
            total_elements = self.store.c_buf._tail // self.store.c_stride
            self.update_sentinels(k_sentinels=max(1, min(100, total_elements)))
            if self.sentinels_matrix is None or len(self.sentinels_ids) == 0:
                raise ValueError("O banco de dados está vazio. Adicione vetores antes de realizar buscas.")
        
        # Normalizar query para similaridade de cosseno
        q_norm = np.linalg.norm(query)
        q_vec = query / q_norm if q_norm > 0 else query
        
        # 1. Coarse Quantization: Busca Matricial Rápida em RAM nas Sentinelas
        sims = np.dot(self.sentinels_matrix, q_vec)
        top_sentinels = np.argsort(-sims)[:n_entry_points]
        
        # 2. Entry points iniciais obtidos a partir das sentinelas correspondentes
        candidates = [self.sentinels_ids[idx] for idx in top_sentinels]
        visited = set(candidates)
        
        best_node = candidates[0]
        best_sim = np.dot(q_vec, self.store.read_vector(best_node))

        # 3. Waggle Dance local (Beam Search) operando no disco
        for _ in range(20):
            next_candidates = []
            for curr in candidates:
                meta = self.store.read_cell_meta(curr)
                neighs = self.store.read_neighbors(meta)
                for n in neighs:
                    if n not in visited:
                        visited.add(n)
                        sim = np.dot(q_vec, self.store.read_vector(n))
                        next_candidates.append((n, sim))
                        if sim > best_sim:
                            best_sim = sim
                            best_node = n

            if not next_candidates: break
            next_candidates.sort(key=lambda x: x[1], reverse=True)
            candidates = [n for n, s in next_candidates[:beam_width]]

        return best_node

In [10]:
import numpy as np

def fix_and_test_knn():
    N, D = 2000, 64
    X = np.random.randn(N, D).astype(np.float32)
    X /= np.linalg.norm(X, axis=1, keepdims=True)

    print("Calculando Ground Truth e Indexando...")
    sim_matrix = np.dot(X, X.T)

    store = DiskHiveStore("hive_fixed", D)
    for i in range(N):
        v_off = store.append_vector(X[i])
        # Aumentado para 20 vizinhos para melhor conectividade
        real_neighbors = np.argsort(-sim_matrix[i])[1:21]
        n_off = store.append_graph_edges(real_neighbors)
        store.write_cell_meta(i, v_off, n_off, len(real_neighbors))

    brain = HiveBrain(store)
    brain.update_sentinels(300) # Mais hubs

    test_indices = np.random.choice(N, 20)
    hits = 0
    for idx in test_indices:
        res = brain.search(X[idx], beam_width=10)
        if res == idx or np.dot(X[res], X[idx]) > 0.99:
            hits += 1

    print(f"Nova Acurácia (Beam Search + 20 Neighbors): {hits/20*100}%")
    # CORREÇÃO: Fechar buffers explicitamente para liberar arquivos
    store.close()

fix_and_test_knn()

Calculando Ground Truth e Indexando...
Nova Acurácia (Beam Search + 20 Neighbors): 100.0%


### 🐝 Mini RAG (Retrieval-Augmented Generation)
Nesta seção, vamos usar o motor de busca `Hive` para recuperar documentos relevantes e responder a perguntas do usuário.

In [13]:
# 1. Base de Conhecimento
knowledge_base = [
    "As abelhas operárias vivem cerca de 6 semanas no verão e realizam todo o trabalho na colmeia.",
    "A abelha rainha é a única fêmea fértil e pode colocar até 2.000 ovos por dia.",
    "O mel é produzido a partir do néctar das flores, que as abelhas processam com enzimas.",
    "A dança do requebrado (waggle dance) é como as abelhas comunicam a localização de comida.",
    "Os zangões são os machos da colmeia e sua única função é acasalar com a rainha.",
    "Uma colmeia saudável pode conter entre 20.000 a 60.000 abelhas."
]

# 2. Simple Vocabulary-based Embedding (TF)
import re
def tokenize(text):
    return re.findall(r'\w+', text.lower())

vocab = sorted(list(set(tokenize(" ".join(knowledge_base)))))
word_to_idx = {word: i for i, word in enumerate(vocab)}

def get_embedding(text, dim=len(vocab)):
    vec = np.zeros(dim, dtype=np.float32)
    for word in tokenize(text):
        if word in word_to_idx:
            vec[word_to_idx[word]] += 1.0
    if np.linalg.norm(vec) > 0:
        vec /= np.linalg.norm(vec)
    return vec

# 3. Indexar Documentos
D = len(vocab)
rag_store = DiskHiveStore("hive_rag", D)
doc_map = {}

embeddings = [get_embedding(doc) for doc in knowledge_base]

print(f"Indexando {len(knowledge_base)} documentos com vocabulário de tamanho {D}...")
for i, doc in enumerate(knowledge_base):
    v_off = rag_store.append_vector(embeddings[i])
    others = [j for j in range(len(knowledge_base)) if j != i]
    n_off = rag_store.append_graph_edges(others)
    rag_store.write_cell_meta(i, v_off, n_off, len(others))
    doc_map[i] = doc

rag_brain = HiveBrain(rag_store)
rag_brain.update_sentinels(k_sentinels=5)
print("RAG Atualizado com busca por palavras-chave!")

Indexando 6 documentos com vocabulário de tamanho 65...
RAG Atualizado com busca por palavras-chave!


In [15]:
def ask_hive_rag(question):
    # Converte pergunta em vetor
    q_vec = get_embedding(question)

    # Recupera o documento mais próximo usando o motor Hive
    best_doc_id = rag_brain.search(q_vec, beam_width=10)
    context = doc_map[best_doc_id]

    print(f"Pergunta: {question}")
    print(f"Contexto Recuperado: {context}")
    print("--- Resposta Final ---")
    return f"Com base nos dados da colmeia: {context}"

# Testando com termos mais específicos para a busca TF
print(ask_hive_rag("Qual a função dos machos na colmeia?"))
print("\n")
# Adicionando 'comunicam' para ajudar o vetor TF a encontrar a 'dança'
print(ask_hive_rag("Como as abelhas comunicam a localização de comida e flores?"))
# CORREÇÃO: Liberar os arquivos do RAG também ao final
rag_store.close()

Pergunta: Qual a função dos machos na colmeia?
Contexto Recuperado: Os zangões são os machos da colmeia e sua única função é acasalar com a rainha.
--- Resposta Final ---
Com base nos dados da colmeia: Os zangões são os machos da colmeia e sua única função é acasalar com a rainha.


Pergunta: Como as abelhas comunicam a localização de comida e flores?
Contexto Recuperado: A dança do requebrado (waggle dance) é como as abelhas comunicam a localização de comida.
--- Resposta Final ---
Com base nos dados da colmeia: A dança do requebrado (waggle dance) é como as abelhas comunicam a localização de comida.
